# 06 FID 평가

학습된 체크포인트에서 FID를 계산합니다.
`launch_eval_fid`는 모든 GPU에서 샘플을 병렬로 생성하고 main process에서 FID를 계산합니다.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_eval_fid

cfg = load_cfg("configs/cifar10.yaml")
checkpoint_path = "outputs/checkpoints/cifar10_full_sdd_last.pt"

launch_eval_fid(
    cfg,
    checkpoint_path=checkpoint_path,
    num_samples=2048,      # 총 fake 샘플 수
    num_processes=None,    # GPU 자동 감지
)


In [ ]:
# baseline vs SDD 비교
from src.experiments import load_cfg, deep_update, launch_eval_fid

cfg_b = load_cfg("configs/cifar10.yaml")
cfg_b = deep_update(cfg_b, {
    "sdd": {"enabled": False, "lambda_distill": 0.0,
            "use_centering": False, "use_sharpening": False,
            "use_gating": False, "use_projection_head": False},
})

print("\n▶ Baseline FID:")
launch_eval_fid(cfg_b, "outputs/checkpoints/cifar10_baseline_mse_only_last.pt",
                num_samples=2048, num_processes=None)

print("\n▶ Full SDD FID:")
launch_eval_fid(load_cfg("configs/cifar10.yaml"),
                "outputs/checkpoints/cifar10_full_sdd_last.pt",
                num_samples=2048, num_processes=None)
